# 2-step RAG
The initial code is mostly based on langchain documentation
https://docs.langchain.com/oss/python/langchain/rag#rag-chains

## Load
I will need to use OCR because the provided pdf is scanned, therefore it cannot be chunked. 

In [2]:
from langchain_community.document_loaders import UnstructuredPDFLoader

# This will use OCR to read the text from the images/scans
loader = UnstructuredPDFLoader(
    "../data/raw/exhibit_1.pdf", 
    mode="single", # previously elements, but it returned caption instead of article
    strategy="ocr_only" # Forces OCR
)
raw_documents = loader.load()

## Chunking
Breaking 11 pages into smaller pieces

In [3]:
#CHUNKING
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)#from 500,100
chunks = text_splitter.split_documents(raw_documents)

# Let's see what happened
print(f"You now have {len(chunks)} pieces of text ready for your collection.")

You now have 31 pieces of text ready for your collection.


## Database
for chunking, indexing

In [4]:
#vectorstore setup
import chromadb
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="thesis_project")
# Chroma needs: Unique IDs for every chunk and text itself

idsList = [f"id_{index}" for index in range(len(chunks))]
docsList= [doc.page_content for doc in chunks]

collection.add(ids=idsList, documents=docsList)
#success message
print(f"Successfully indexed {collection.count()} chunks into ChromaDB.")

Successfully indexed 31 chunks into ChromaDB.


# Retrieval

In [ ]:
query = "What is Zostera marina?" # Example question

# Asks Chroma to find the 3 most relevant chunks
results = collection.query(
    query_texts=[query],
    n_results=5
)

# This is the retrieved context
retrieved_context = " ".join(results['documents'][0])
print("Retrieved Context:", retrieved_context[:200], "...")

Retrieved Context: Note: This species often occurs in exposed area. Phyllospadix japonica Makino

Plants 40 cm high. Leaves obtuse in tip, three in vein, 0.15 cm broad, di- and tridentate with spine cells in mar- gin, w ...


# Generation
I am using openai appi with gpt5-nano respond based on the query and retrieved knowledge.



The taxonomy paper emphasizes that when instructions lack clarity, LLM responses become unfocused, so being explicit about demand (what format, what scope, what to do when context is missing) is what separates a vague prompt from one I can systematically optimize.
Their "Profile and Instruction" category breaks a good prompt into three pieces: personality information (the marine scientist role), task information (intent + domain + demand), and demonstration information (few-shot examples).

In [7]:
print(retrieved_context)

Note: This species often occurs in exposed area. Phyllospadix japonica Makino

Plants 40 cm high. Leaves obtuse in tip, three in vein, 0.15 cm broad, di- and tridentate with spine cells in mar- gin, with one or two sponge layers in cross section view. Seeds liver-shaped, with two lobes. Plants epilithic in the exposed littoral and reproductive in spring.

Note: This species often occurs in the sheltered area. Zostera Linnaeus, 1753

The genus Zostera is typified as Z. marina Linnaeus and includes 12 species in the world. Phillips and Menez (1988) reviewed the Zostera members. in the world. The members often grow in the muddy and shel-

tered area. Key to the species 1. Plants small, leaves narrow and veins three bocce eee secu eee ceaeceu secu eeuseaeesaecepecaeeneties Z. japonica 1. Plants tall, leaves wide, veins more than five -:---: 2 2. Leave tips mucronate and seeds with rough testa bocce eec cae eeeaeeceasecaeeeeeeeeaaeeeeeseanceeeseeeeeas Z. marina 2. Leave tips obtuse and seed

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
  api_key=os.getenv("OPENAI_API_KEY")
  
)
prompt = f"""
Role: You are an experienced marine biologist and algae cultivation specialist with deep expertise in seagrass ecology, microalgae biotechnology, 
and industrial algae applications.
You prioritize scientific precision, cite 
specific data when available, and clearly distinguish between established 
findings and uncertainties.
Task: Given retrieved passages from scientific literature on algae and 
marine biology, answer the user's question following these steps:

1. First, assess which of the provided passages are relevant to the question 
   and briefly note why.
2. Synthesize information from the relevant passages into a coherent answer.
3. If passages contain conflicting information, acknowledge the conflict and 
   explain both positions.
4. If the provided context is insufficient to fully answer the question, 
   state what is missing.
Domain constraint: Focus on the user's specific industry context.
Output format: Brief paragraph with a traceable callback to the used chunk. 

Keep your answer grounded strictly in the provided context. Do not introduce 
external knowledge beyond what is given. Match your answer's depth to the question's complexity: give concise 
answers to simple questions, detailed analysis to complex ones.

Context: {retrieved_context}

Question: {query}
Answer:"""
response = client.chat.completions.create(
  model="gpt-5-nano",
  messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content);


Zostera marina Linnaeus is the type species of the genus Zostera, a submerged seagrass (marine angiosperm) commonly found on muddy substrates in the sublittoral zone, with reproduction concentrated in summer. In the provided material, Z. marina is described as plants about 34 cm tall, submerged, with leaves obtuse and three veins, around 0.2 cm broad; seeds are about 2.5–0.7 cm long and 0.2 cm broad, with a rough seed testa. (Zostera marina section in the Cho & Boo marine flora text)


# Reranking
